# Week 4 Logistic Regression & Feature Scaling

**Datasets:** UCL Online Retail II

## Topics
1. **Binary classification (UCL)** — predict repeat buyers (frequency ≥ 2) from behavioral features.
2. **Feature scaling** — compare unscaled vs. StandardScaler logistic regression.


## Data loading and cleaning

Load customer-level RFM features from all three datasets.


In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

np.random.seed(42)
DATA_DIR = Path(".")

orders = pd.read_csv(DATA_DIR / "olist_Brazilian" / "olist_orders_dataset.csv")
reviews = pd.read_csv(DATA_DIR / "olist_Brazilian" / "olist_order_reviews_dataset.csv")
items = pd.read_csv(DATA_DIR / "olist_Brazilian" / "olist_order_items_dataset.csv")
payments = pd.read_csv(DATA_DIR / "olist_Brazilian" / "olist_order_payments_dataset.csv")
customers = pd.read_csv(DATA_DIR / "olist_Brazilian" / "olist_customers_dataset.csv")

pay = payments.groupby("order_id").agg(
    payment_value=("payment_value", "sum"),
    payment_installments=("payment_installments", "max"),
    n_payments=("payment_sequential", "max"),
).reset_index()
itm = items.groupby("order_id").agg(
    total_price=("price", "sum"),
    freight_value=("freight_value", "sum"),
    n_items=("order_item_id", "count"),
).reset_index()
rev = reviews.groupby("order_id").agg(review_score=("review_score", "mean")).reset_index()

olist_df = (
    orders.merge(rev, on="order_id")
    .merge(itm, on="order_id")
    .merge(pay, on="order_id")
    .merge(customers[["customer_id", "customer_unique_id", "customer_state"]], on="customer_id")
)
olist_df["order_purchase_timestamp"] = pd.to_datetime(olist_df["order_purchase_timestamp"])
olist_df["order_month"] = olist_df["order_purchase_timestamp"].dt.month
olist_df["delivery_days"] = (
    pd.to_datetime(olist_df["order_delivered_customer_date"], errors="coerce")
    - olist_df["order_purchase_timestamp"]
).dt.days
olist_df["delivery_days"] = olist_df["delivery_days"].fillna(olist_df["delivery_days"].median())
olist_df = olist_df.dropna(subset=["review_score", "payment_value", "total_price"])

ucl_raw = pd.read_excel(DATA_DIR / "ucl_dataset.xlsx")
ucl_raw = ucl_raw.rename(columns={"Invoice": "InvoiceNo", "Price": "UnitPrice", "Customer ID": "CustomerID"})
ucl_raw["InvoiceDate"] = pd.to_datetime(ucl_raw["InvoiceDate"])
ucl_raw = ucl_raw.dropna(subset=["CustomerID"])
ucl_raw = ucl_raw[~ucl_raw["InvoiceNo"].astype(str).str.startswith("C", na=False)]
ucl_raw = ucl_raw[(ucl_raw["Quantity"] > 0) & (ucl_raw["UnitPrice"] > 0)]
ucl_raw["LineTotal"] = ucl_raw["Quantity"] * ucl_raw["UnitPrice"]
anchor = ucl_raw["InvoiceDate"].max() + pd.Timedelta(days=1)
ucl_df = (
    ucl_raw.groupby("CustomerID")
    .agg(
        recency=("InvoiceDate", lambda x: (anchor - x.max()).days),
        frequency=("InvoiceNo", "nunique"),
        monetary=("LineTotal", "sum"),
        avg_basket=("LineTotal", "mean"),
        n_items=("Quantity", "sum"),
    )
    .reset_index()
)
ucl_df["log_monetary"] = np.log1p(ucl_df["monetary"])
ucl_df["log_frequency"] = np.log1p(ucl_df["frequency"])

TAOBAO_PATH = DATA_DIR / "TaobaoUserBehavior.csv"
user_set = set()
chunks = []
for chunk in pd.read_csv(
    TAOBAO_PATH,
    header=None,
    names=["user_id", "item_id", "category_id", "behavior", "timestamp"],
    chunksize=500_000,
):
    chunk = chunk[chunk["user_id"].isin(user_set) | (len(user_set) < 50000)]
    if len(user_set) < 50000:
        for u in chunk["user_id"].unique():
            if len(user_set) >= 50000:
                break
            user_set.add(u)
    chunk = chunk[chunk["user_id"].isin(user_set)]
    if len(chunk):
        chunks.append(chunk)
    if len(user_set) >= 50000 and sum(len(c) for c in chunks) > 200_000:
        break

taobao = pd.concat(chunks, ignore_index=True)
taobao_df = (
    taobao.groupby("user_id")
    .agg(
        n_events=("behavior", "count"),
        n_pv=("behavior", lambda x: (x == "pv").sum()),
        n_cart=("behavior", lambda x: (x == "cart").sum()),
        n_fav=("behavior", lambda x: (x == "fav").sum()),
        n_buy=("behavior", lambda x: (x == "buy").sum()),
        n_categories=("category_id", "nunique"),
        n_items=("item_id", "nunique"),
    )
    .reset_index()
)
taobao_df["cart_rate"] = taobao_df["n_cart"] / taobao_df["n_events"].clip(lower=1)
taobao_df["converted"] = (taobao_df["n_buy"] > 0).astype(int)

print(f"Olist orders: {len(olist_df):,}")
print(f"UCL customers: {len(ucl_df):,}")
print(f"Taobao users: {len(taobao_df):,}")


# Define repeat-buyer target

Label loyal customers as those with at least two distinct orders.


In [ ]:
ucl_df = ucl_df.copy()
ucl_df["loyal"] = (ucl_df["frequency"] >= 2).astype(int)

feats = ["recency", "avg_basket", "n_items"]
X = ucl_df[feats].values
y = ucl_df["loyal"].values

print(f"Repeat buyers: {y.sum():,} ({100 * y.mean():.1f}%)")
print(f"One-time buyers: {(1 - y).sum():,} ({100 * (1 - y.mean()):.1f}%)")


# Logistic regression without scaling


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

unscaled = LogisticRegression(max_iter=1000).fit(X_train, y_train)
pred_u = unscaled.predict(X_test)
proba_u = unscaled.predict_proba(X_test)[:, 1]

print(f"Unscaled accuracy: {accuracy_score(y_test, pred_u):.4f}")
print(f"Unscaled ROC-AUC: {roc_auc_score(y_test, proba_u):.4f}")
print(f"Unscaled F1: {f1_score(y_test, pred_u):.4f}")


# Logistic regression with StandardScaler


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=1000)),
])
scaled = pipe.fit(X_train, y_train)
pred_s = scaled.predict(X_test)
proba_s = scaled.predict_proba(X_test)[:, 1]

print(f"Scaled accuracy: {accuracy_score(y_test, pred_s):.4f}")
print(f"Scaled ROC-AUC: {roc_auc_score(y_test, proba_s):.4f}")
print(f"Scaled F1: {f1_score(y_test, pred_s):.4f}")


Recency is measured in days while basket and item counts are on different scales. StandardScaler aligns magnitudes so gradient-based fitting is not dominated by one feature.


# Confusion matrix


In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt

cm = confusion_matrix(y_test, pred_s)
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(["One-time", "Repeat (2+ orders)"])
ax.set_yticklabels(["One-time", "Repeat (2+ orders)"])
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i, j], ha="center", va="center", color="black")
ax.set_title("Week 4: Logistic regression (UCL repeat vs one-time buyers)")
plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.show()
